# AutoMotion Mobile V9 — Renderizador Colab

Este notebook faz a parte pesada do AutoMotion **fora do celular**.

**Entrada:** vídeo original + `automotion-astronomia-project.json`.

**Saída:** MP4 renderizado com FFmpeg.

A análise e os timestamps continuam no navegador; o Colab faz a composição final.


In [ ]:
# @title 1) Enviar vídeo + projeto JSON
from google.colab import files
import os, json, pathlib, subprocess, re, math

uploaded = files.upload()
print("Arquivos recebidos:", list(uploaded.keys()))
video_candidates = [n for n in uploaded if pathlib.Path(n).suffix.lower() in {'.mp4','.mov','.webm','.m4v','.avi','.mkv'}]
json_candidates = [n for n in uploaded if pathlib.Path(n).suffix.lower()=='.json']
if not video_candidates or not json_candidates:
    raise RuntimeError("Envie o vídeo original e o automotion-astronomia-project.json juntos.")
VIDEO = video_candidates[0]
PROJECT_FILE = json_candidates[0]
with open(PROJECT_FILE, 'r', encoding='utf-8') as f:
    PROJECT = json.load(f)
print("Vídeo:", VIDEO)
print("Projeto:", PROJECT_FILE)
print("Claims ativos:", len(PROJECT.get('claims', [])))


## 2) Configuração

O padrão é 30 FPS, H.264 e `veryfast`. Use o teste de 30 segundos para validar um projeto antes do render completo.


In [ ]:
# @title 2) Configuração
TESTE_30S = False  # @param {type:"boolean"}
MAX_DIM = 1920      # @param {type:"slider", min:720, max:3840, step:120}
FPS = int(PROJECT.get('render', {}).get('fps', 30) or 30)
CRF = int(PROJECT.get('render', {}).get('crf', 20) or 20)
PRESET = PROJECT.get('render', {}).get('preset', 'veryfast') or 'veryfast'
AUDIO_BITRATE = PROJECT.get('render', {}).get('audio_bitrate', '160k') or '160k'
STYLE = PROJECT.get('render', {}).get('style', 'cinematic') or 'cinematic'
POSITION = PROJECT.get('render', {}).get('position', 'auto') or 'auto'
DURATION = float(PROJECT.get('video', {}).get('duration_seconds') or 0)
CLAIMS = [c for c in PROJECT.get('claims', []) if c.get('enabled', True) is not False]
END_LIMIT = min(30.0, DURATION) if TESTE_30S and DURATION else (30.0 if TESTE_30S else None)
print(f"FPS: {FPS} | CRF: {CRF} | preset: {PRESET} | estilo: {STYLE} | posição: {POSITION}")
print(f"Claims: {len(CLAIMS)} | teste 30s: {TESTE_30S}")


In [ ]:
# @title 3) Montar as visualizações FFmpeg
FONT_BOLD = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
FONT_REG = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
if not os.path.exists(FONT_BOLD): FONT_BOLD = "DejaVu Sans:style=Bold"
if not os.path.exists(FONT_REG): FONT_REG = "DejaVu Sans"

def ff_escape(s):
    return (str(s or '').replace('\\', r'\\').replace(':', r'\\:').replace("'", r"\\'").replace('%', r'\\%').replace(',', r'\\,'))
def num(v, default=0):
    try: return float(v)
    except: return float(default)
def clamp(v, lo, hi): return max(lo, min(hi, v))
def theme_values(style):
    if style == 'minimal': return dict(panel='0x071018E6', accent='0xFFFFFFFF', text='0xFFFFFFFF', muted='0xB9C2CEFF')
    if style == 'hud': return dict(panel='0x020912EA', accent='0x4DDFFFFF', text='0xDFFBFFFF', muted='0x83A7B5FF')
    return dict(panel='0x040913E0', accent='0x79C0FFFF', text='0xF3F8FFFF', muted='0x9BB4CFFF')
TH = theme_values(STYLE)

def x_expr(pos, typ, start):
    slide = f"(52*if(gte(t\,{start+.45:.3f})\,0\,1-(t-{start:.3f})/0.45))"
    if pos == 'right' or (pos == 'auto' and typ in ('ratio','percentage')): return f"w-w*0.72-w*0.04+{slide}"
    if pos == 'center': return "(w-w*0.72)/2"
    return f"w*0.04-{slide}"

def add_claim(filters, c):
    start, end = num(c.get('start')), num(c.get('end'), num(c.get('start'))+1)
    if end <= start: end = start + 1
    typ = c.get('type','number')
    x = x_expr(POSITION, typ, start)
    w = "iw*0.72"
    en = f"between(t\\,{start:.3f}\\,{end:.3f})"
    a = f"if(lt(t\\,{start:.3f}),0,if(lt(t\\,{start+.45:.3f}),(t-{start:.3f})/0.45,if(gt(t\\,{max(start,end-.35):.3f}),({end:.3f}-t)/0.35,1)))"
    label = ff_escape(c.get('label','Informação').upper())
    headline = ff_escape(c.get('headline',''))
    detail0 = str(c.get('detail',''))
    detail = ff_escape(detail0[:68] + ('…' if len(detail0)>68 else ''))
    filters += [
        f"drawbox=x={x}:y=ih*0.10:w={w}:h=ih*0.24:color={TH['panel']}:t=fill:enable='{en}'",
        f"drawtext=fontfile='{FONT_BOLD}':text='{label}':x={x}+18:y=h*0.10+18:fontsize=18:fontcolor={TH['accent']}:alpha='{a}':enable='{en}'",
        f"drawtext=fontfile='{FONT_BOLD}':text='{headline}':x={x}+18:y=h*0.10+72:fontsize=40:fontcolor={TH['text']}:alpha='{a}':enable='{en}'",
        f"drawtext=fontfile='{FONT_REG}':text='{detail}':x={x}+18:y=h*0.10+104:fontsize=16:fontcolor={TH['muted']}:alpha='{a}':enable='{en}'",
    ]
    base = "h*0.10+126"; x0=f"{x}+18"; avail=f"({w}-36)"
    frac = clamp(abs(num(c.get('value'))),0,100)/100.0
    if typ == 'ratio':
        filters += [
            f"drawbox=x={x0}:y={base}:w={avail}:h=8:color=0xFFFFFF26:t=fill:enable='{en}'",
            f"drawbox=x={x0}:y={base}:w={avail}*min(1\\,{max(abs(num(c.get('value'))),1):.6f}/100):h=8:color={TH['accent']}:t=fill:enable='{en}'",
            f"drawtext=fontfile='{FONT_REG}':text='1x':x={x0}:y={base}+23:fontsize=14:fontcolor={TH['muted']}:alpha='{a}':enable='{en}'"
        ]
    elif typ in ('percentage','temperature','number','speed'):
        if typ=='percentage': q=clamp(num(c.get('value')),0,100)/100
        elif typ=='temperature': q=clamp((num(c.get('value'))+200)/1400,0,1)
        elif typ=='speed': q=clamp(math.log10(max(abs(num(c.get('value'))),1))/6,0,1)
        else: q=clamp(abs(num(c.get('value')))/100,0,1)
        filters += [
            f"drawbox=x={x0}:y={base}:w={avail}:h=9:color=0xFFFFFF26:t=fill:enable='{en}'",
            f"drawbox=x={x0}:y={base}:w={avail}*{q:.6f}*min(max((t-{start:.3f})/0.65\\,0)\\,1):h=9:color={TH['accent']}:t=fill:enable='{en}'"
        ]
    elif typ == 'distance':
        filters += [
            f"drawbox=x={x0}:y={base}:w={avail}:h=2:color=0xFFFFFF33:t=fill:enable='{en}'",
            f"drawbox=x={x0}+({avail})*min(max((t-{start:.3f})/0.70\\,0)\\,1):y={base}-4:w=9:h=9:color={TH['accent']}:t=fill:enable='{en}'"
        ]
    elif typ in ('time','timeScale'):
        filters += [
            f"drawbox=x={x0}:y={base}:w={avail}:h=4:color=0xFFFFFF26:t=fill:enable='{en}'",
            f"drawbox=x={x0}:y={base}:w={avail}*min(max((t-{start:.3f})/0.80\\,0)\\,1):h=4:color={TH['accent']}:t=fill:enable='{en}'"
        ]

filters=[]
for c in CLAIMS[:60]: add_claim(filters,c)
VF = ','.join(filters) if filters else 'null'
print("Operações FFmpeg:", len(filters), "| claims:", min(len(CLAIMS),60))


In [ ]:
# @title 4) Renderizar MP4

def render():
    out='automotion_final.mp4'
    base_scale=(f"scale=w='if(gt(iw\\,ih)\\,min(iw\\,{MAX_DIM}),-2)':h='if(gt(iw\\,ih),-2,min(ih,{MAX_DIM}))'")
    final_vf = base_scale + (','+VF if VF != 'null' else '')
    cmd=['ffmpeg','-y','-hide_banner','-loglevel','error']
    if END_LIMIT is not None: cmd += ['-t', str(END_LIMIT)]
    cmd += ['-i',VIDEO,'-vf',final_vf,'-r',str(FPS),'-c:v','libx264','-preset',PRESET,'-crf',str(CRF),'-pix_fmt','yuv420p','-c:a','aac','-b:a',AUDIO_BITRATE,'-movflags','+faststart','-progress','pipe:1','-nostats',out]
    print('Executando FFmpeg…')
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,bufsize=1)
    last=-2.0
    while True:
        line=p.stdout.readline()
        if not line:
            if p.poll() is not None: break
            continue
        line=line.strip()
        if line.startswith('out_time_ms='):
            try:
                sec=float(line.split('=',1)[1])/1e6
                total=END_LIMIT or DURATION or 0
                if total:
                    pct=sec/total*100
                    if pct-last>=2:
                        print(f'Progresso: {min(100,pct):.1f}% | {sec:.1f}s',flush=True); last=pct
            except: pass
    err=p.stderr.read(); rc=p.wait()
    if rc: raise RuntimeError('FFmpeg falhou:\n'+err[-7000:])
    print('✅ Render concluído:',out)
    return out
OUTPUT=render()


## 5) Baixar

O MP4 final está no ambiente do Colab.


In [ ]:
# @title 5) Baixar MP4
from google.colab import files
files.download(OUTPUT)


In [ ]:
# @title Diagnóstico do arquivo final
p=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size','-of','default=noprint_wrappers=1',OUTPUT],capture_output=True,text=True)
print(p.stdout or 'Sem dados.')
